# Milestone - Unit 3: the lexical path joins the recommender -- books whose words match what you have read

## 1. Load the catalog, the interactions, and the keyword documents

The same generated substrate Units 1 and 2 ran on, plus the per-book keyword bags `keywords.csv.gz`
(recsys-004): one space-joined document per book, with real repetition and length variation.
`bookrec.generated_dir()` reports a clear error if `recsys/data/generated/` has not been generated yet.

In [ ]:
import json
from collections import Counter, defaultdict

import bookrec
import pandas as pd

K = 10
RANDOM_SEED = 0

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = cold["cold_readers"]

interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")

doc_lengths = [len(keywords[item_id].split()) for item_id in catalog_ids]
{
    "books": len(catalog_ids),
    "keyword_documents": len(keywords),
    "interactions": len(rows),
    "shortest_document": min(doc_lengths),
    "longest_document": max(doc_lengths),
}

## 2. Build, fit, and register the lexical path next to popularity and the random floor

`LexicalRetrievalPath` takes its keyword corpus at construction (like `RandomRetrievalPath` takes its
item ids), so `fit(interactions, catalog=...)` keeps the exact `RetrievalPath` signature: it ignores
the interaction log and builds a BM25 index over the catalog's keyword documents. It is the first
*reader-dependent* path in the system -- a reader's query is the keyword tokens of the books in
`context["seen"]`, and it retrieves unseen books whose words match. The registry now holds three
paths with different signals: chance, crowd counts, and content.

In [ ]:
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=RANDOM_SEED)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords, k1=1.2, b=0.75).fit(rows, catalog=catalog_ids)

registry = bookrec.PathRegistry()
registry.register(random_floor)
registry.register(popularity)
registry.register(lexical)

index = lexical.artifact()
{
    "registered": [path.artifact_name() for path in registry],
    "indexed_documents": len(index["item_ids"]),
    "vocabulary_size": len(index["vocab"]),
    "average_document_length": round(index["avgdl"], 2),
    "k1": index["k1"],
    "b": index["b"],
}

## 3. Score it on the frozen `val` scoreboard against the floor and against popularity

Same scoreboard as Units 1 and 2: positive `val` rows, `k = 10`, cold readers excluded, a reader's
positive `train` items form the seen set (the lexical query) and are never recommended back.

In [ ]:
boards = {
    path.name: bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    )
    for path in registry
}

floor_hit = boards["random"].hit_rate_at_k
popularity_hit = boards["popularity"].hit_rate_at_k
lexical_hit = boards["lexical"].hit_rate_at_k

{
    "scored_readers": boards["lexical"].readers,
    "random_floor_hit_at_10": round(floor_hit, 4),
    "popularity_hit_at_10": round(popularity_hit, 4),
    "lexical_hit_at_10": round(lexical_hit, 4),
    "lexical_over_floor": round(lexical_hit / floor_hit, 1),
    "lexical_over_popularity": round(lexical_hit / popularity_hit, 2),
}

Lexical BM25 lands around 0.158 hit@10: roughly 13x the random floor (~0.012) and about 1.46x the
popularity path (~0.108). Read that honestly. It is a strong **content candidate source** -- the first
path that recommends different books to different readers -- and it is comparable to what simple
genre matching achieves on this data. It still sits well below the collaborative and latent paths
of Units 4 and 5, and we make no claim that BM25 beats the TF-IDF cosine scorer derived in the
lesson: on these short, lightly repeating keyword bags the two are about on par.

## 4. One reader's keyword profile, and why each recommendation was retrieved

Pick a reader deterministically (the lowest-id warm reader), pool the keyword tokens of the books
they have read into a profile, and ask the lexical path for its top 10. Every recommendation can be
explained by the words it shares with the profile -- the explanation lexical retrieval gives for free
and that later, opaque paths will not.

In [ ]:
seen_by = defaultdict(set)
for row in rows:
    if row["split"] == "train" and row["label"] == 1 and row["item_id"] in catalog:
        seen_by[row["reader_id"]].add(row["item_id"])

cold_set = set(cold_readers)
reader_id = min(r for r in seen_by if r not in cold_set)
seen = seen_by[reader_id]

profile = Counter()
for item_id in sorted(seen):
    profile.update(keywords[item_id].split())

candidates = lexical.retrieve(reader_id, {"seen": seen}, K)
recommendations = bookrec.rank(candidates, n=K, exclude=seen)

explained = []
for candidate in recommendations:
    book_words = set(keywords[candidate.item_id].split())
    # Shared words, the ones the reader has seen most often first.
    shared = [word for word, _ in profile.most_common() if word in book_words]
    book = catalog[candidate.item_id]
    explained.append(
        {
            "item_id": candidate.item_id,
            "title": book.title,
            "genres": book.fields["genres"],
            "bm25": round(candidate.score, 1),
            "shared": len(shared),
            "top_shared_words": " ".join(shared[:5]),
        }
    )

print("reader", reader_id, "has read", len(seen), "books:")
for item_id in sorted(seen):
    print("  ", item_id, catalog[item_id].title, "|", catalog[item_id].fields["genres"])
print("top profile words:", " ".join(f"{word}x{n}" for word, n in profile.most_common(10)))
print("ranked ids:", [candidate.item_id for candidate in recommendations])
pd.DataFrame(explained)

Every retrieved book shares several profile words, and the strongest matches are the books that
repeat the rarer words the reader has seen most -- the IDF weighting the lesson derived. The genres
line up loosely with the reader's history (the generator leaks genre into keywords, which is why
lexical and genre matching score alike). Unit 4 adds the signal this path cannot see: what *other*
readers with the same history went on to like.